# ViT & Modern Generative AI

This notebook accompanies the **ML Viz** lesson on Vision Transformers, StyleGAN, CycleGAN, and the Stable Diffusion latent pipeline.
We implement the core ideas in pure NumPy — no GPU or pretrained weights required.

**Companion lesson:** https://ml-viz-ruby.vercel.app/courses/generative-models/06-vit-and-modern-genai

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive. Changes to this view are not saved.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# Dark matplotlib style matching the ML Viz design
plt.rcParams['figure.facecolor'] = '#0f1117'
plt.rcParams['axes.facecolor']   = '#1a1d27'
plt.rcParams['text.color']       = 'white'
plt.rcParams['axes.labelcolor']  = '#94a3b8'
plt.rcParams['xtick.color']      = '#94a3b8'
plt.rcParams['ytick.color']      = '#94a3b8'
plt.rcParams['axes.edgecolor']   = '#2e3347'
plt.rcParams['axes.spines.top']  = False
plt.rcParams['axes.spines.right']= False

rng = np.random.default_rng(42)
print('NumPy', np.__version__)

## Intuition — the building blocks of modern image AI

Five mechanisms power today's image systems, and each is small enough to verify by hand. **ViT patch
embedding** chops an image into 16×16 patches and linearly projects each into a token — turning vision
into a sequence problem the transformer already solves. **Classifier-free guidance** steers a diffusion
model toward its prompt by extrapolating past the conditional prediction:
`ε = (1+w)·ε_cond − w·ε_uncond`. **Cycle consistency** trains unpaired translators (horses↔zebras) by
demanding round trips return the original. **Latent diffusion** (Stable Diffusion) runs the expensive
denoising in a VAE's compressed latent space — ~48× fewer values than pixels. We implement each and
verify the arithmetic that makes them work.

## 1. Vision Transformer — Patch Embedding

ViT splits an image into $P \times P$ patches, flattens each, and projects linearly to $d$ dimensions.

**Worked example:** 224×224 image, $P=16$

- Patches per axis: $224 / 16 = 14$
- Total patches: $14^2 = 196$
- Pixels per patch: $16 \times 16 \times 3 = 768$
- After linear projection: each patch → $d$-dim token

Add 1 [CLS] token → sequence length = **197**.

In [ ]:
def patch_embed(image, patch_size, d_model, seed=0):
    """
    Split image into non-overlapping patches and project each patch linearly.

    Parameters
    ----------
    image      : (H, W, C) float array
    patch_size : int, P
    d_model    : int, embedding dimension
    seed       : int, for reproducible random projection weights

    Returns
    -------
    tokens : (num_patches, d_model)
    """
    H, W, C = image.shape
    P = patch_size
    assert H % P == 0 and W % P == 0, f"Image dimensions must be divisible by patch size P={P}"

    n_h, n_w = H // P, W // P  # patches per axis
    num_patches = n_h * n_w     # total patches

    # Reshape to (n_h, n_w, P, P, C) then flatten each patch → (num_patches, P*P*C)
    patches = image.reshape(n_h, P, n_w, P, C)   # interleave spatial / patch dims
    patches = patches.transpose(0, 2, 1, 3, 4)   # (n_h, n_w, P, P, C)
    flat = patches.reshape(num_patches, P * P * C)  # (num_patches, raw_dim)

    # Learned linear projection W_E  ∈  R^(raw_dim × d_model)  (random init for demo)
    local_rng = np.random.default_rng(seed)
    W_E = local_rng.normal(0, 0.02, (P * P * C, d_model))

    tokens = flat @ W_E  # (num_patches, d_model)
    return tokens


# ── Demo: 224×224 RGB image, ViT-B/16 settings ──────────────────────────────
H, W, C = 224, 224, 3
P, d    = 16, 768

image  = rng.normal(0, 1, (H, W, C)).astype(np.float32)
tokens = patch_embed(image, patch_size=P, d_model=d)

print(f"Image shape         : {image.shape}")
print(f"Patches per axis    : {H//P} × {W//P} = {(H//P)*(W//P)}")
print(f"Raw patch dim (P²C) : {P}×{P}×{C} = {P*P*C}")
print(f"Token matrix shape  : {tokens.shape}")
print(f"After prepending [CLS]: sequence length = {tokens.shape[0] + 1}")
print()

# ── Compare three ViT variants ────────────────────────────────────────────────
configs = [
    dict(name="ViT-B/16", H=224, P=16, d=768,  L=12, heads=12),
    dict(name="ViT-L/16", H=224, P=16, d=1024, L=24, heads=16),
    dict(name="ViT-H/14", H=224, P=14, d=1280, L=32, heads=16),
]

print(f"{'Variant':<12} {'Patches':>8} {'RawDim':>8} {'d':>6} {'SeqLen':>8}")
print("-" * 48)
for cfg in configs:
    n_patches = (cfg['H'] // cfg['P']) ** 2
    raw_dim   = cfg['P'] ** 2 * 3
    seq_len   = n_patches + 1  # +1 for [CLS]
    print(f"{cfg['name']:<12} {n_patches:>8} {raw_dim:>8} {cfg['d']:>6} {seq_len:>8}")

**What to notice:** the entire "vision" part of a ViT is this one function — reshape into `P×P`
patches, flatten each, one shared linear projection. A 224×224 image becomes **196 tokens** of
dimension 768, and from that point on it *is* a transformer input; everything after is the standard
architecture from the transformers course.

## The library way — verify the patch reshape against explicit slicing

The reshape/transpose gymnastics are exactly where silent bugs live (patches scrambled but shapes
fine). The check: extract each patch with explicit `image[i*P:(i+1)*P, j*P:(j+1)*P]` slicing and
assert it matches the vectorized reshape.

In [ ]:
P_t, d_t = 16, 32
img_t = rng.normal(0, 1, (64, 64, 3))
n_side = 64 // P_t

# vectorized route (same as patch_embed's internals)
pt = img_t.reshape(n_side, P_t, n_side, P_t, 3).transpose(0, 2, 1, 3, 4).reshape(n_side*n_side, -1)

# explicit slicing route
manual = np.stack([img_t[i*P_t:(i+1)*P_t, j*P_t:(j+1)*P_t].ravel()
                   for i in range(n_side) for j in range(n_side)])

assert np.allclose(pt, manual), "vectorized patching must equal explicit slicing"
print(f'{n_side*n_side} patches, each {P_t}x{P_t}x3 -> flattened {P_t*P_t*3}')
print('vectorized reshape/transpose == explicit patch slicing ✓')

**What to notice:** the reshape-transpose route reproduces explicit slicing patch-for-patch — the
interleaved `reshape(n_h, P, n_w, P, C).transpose(0,2,1,3,4)` is correct. Get the transpose order
wrong and the shapes still work out while the patches are scrambled; this identity test is the guard.

## 2. Classifier-Free Guidance (CFG)

CFG lets a single diffusion model handle both conditional ($\varepsilon_c$) and unconditional ($\varepsilon_\emptyset$) outputs.
At inference, the combined noise prediction is:

$$\hat{\varepsilon} = (1+w)\,\varepsilon_c - w\,\varepsilon_\emptyset$$

- $w=0$: purely conditional (no guidance boost)
- $w=7.5$: strong guidance — Stable Diffusion default

The difference $\varepsilon_c - \varepsilon_\emptyset$ is the **guidance direction** — the direction in noise space that shifts the sample toward the condition.

In [ ]:
def cfg_output(cond_output, uncond_output, guidance_scale):
    """Classifier-Free Guidance combination.

    Parameters
    ----------
    cond_output   : array, conditional model output ε_c
    uncond_output : array, unconditional model output ε_∅
    guidance_scale: float, w ≥ 0

    Returns
    -------
    combined output of same shape
    """
    w = guidance_scale
    return (1 + w) * cond_output - w * uncond_output


# ── 1-D illustration of how guidance scale shifts output ─────────────────────
# Simulate a 1-D latent: uncond points at 0, cond points at +1
uncond = np.array([0.0])
cond   = np.array([1.0])

w_values = [0, 1, 3, 7.5, 12, 20]
outputs  = [cfg_output(cond, uncond, w)[0] for w in w_values]

fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(w_values, outputs, 'o-', color='#818cf8', linewidth=2, markersize=7)
ax.axhline(cond[0],   linestyle='--', color='#14b8a6', alpha=0.6, label='Conditional output (ε_c = 1)')
ax.axhline(uncond[0], linestyle='--', color='#f43f5e', alpha=0.6, label='Unconditional output (ε_∅ = 0)')
ax.set_xlabel('Guidance scale w')
ax.set_ylabel('Combined CFG output')
ax.set_title('How CFG guidance scale amplifies the conditional direction', color='white')
ax.legend()
plt.tight_layout()
plt.show()

# ── Numeric summary ───────────────────────────────────────────────────────────
print(f"{'w':>6}  {'CFG output':>12}")
print("-" * 22)
for w, out in zip(w_values, outputs):
    note = "  ← SD default" if w == 7.5 else ""
    note = "  ← no boost" if w == 0 else note
    print(f"{w:>6.1f}  {out:>12.3f}{note}")

**What to notice:** CFG **extrapolates**, not interpolates — at `w=0` you get the conditional
prediction, and every increase in `w` pushes the output *further past* it, away from the unconditional
baseline. That's why guidance sharpens prompt adherence (the update exaggerates "what the prompt
changes") and why cranking `w` too high oversaturates and distorts — you're leaving the data manifold.

## 3. CycleGAN — Cycle Consistency Loss

CycleGAN translates images between domains **without paired training data**.
The cycle consistency loss enforces:

$$\mathcal{L}_{cyc} = \mathbb{E}_x\bigl[\|G_{BA}(G_{AB}(x)) - x\|_1\bigr]
  + \mathbb{E}_y\bigl[\|G_{AB}(G_{BA}(y)) - y\|_1\bigr]$$

If a generator collapses (always outputs the same image regardless of input), the reverse generator cannot reconstruct the original, so the cycle loss is large — this penalizes mode collapse.

In [ ]:
def cycle_consistency_loss(x, G_AB, G_BA):
    """
    Compute L1 cycle consistency loss for images in domain A.

    Parameters
    ----------
    x    : (N, ...) array of images from domain A
    G_AB : callable  A → B
    G_BA : callable  B → A

    Returns
    -------
    scalar L1 cycle loss
    """
    x_in_B         = G_AB(x)          # A → B
    x_reconstructed = G_BA(x_in_B)   # B → A  (should recover x)
    return np.mean(np.abs(x_reconstructed - x))


# ── Toy generators: G_AB adds brightness, G_BA subtracts it ──────────────────
# Perfect cycle: G_BA undoes exactly what G_AB did → loss ≈ 0
G_AB_perfect = lambda x: x + 0.5
G_BA_perfect = lambda x: x - 0.5

# Collapsed generator: G_AB always outputs zeros → cycle loss = mean(|x|)
G_AB_collapsed = lambda x: np.zeros_like(x)
G_BA_collapsed = lambda x: np.zeros_like(x)

# Noisy generator: adds small random perturbation
G_AB_noisy = lambda x: x + rng.normal(0, 0.1, x.shape)
G_BA_noisy = lambda x: x - rng.normal(0, 0.1, x.shape)  # approximate inverse

# Create a batch of 32 fake 16-channel "images" (simplified for illustration)
x_batch = rng.normal(0, 1, (32, 16))

loss_perfect   = cycle_consistency_loss(x_batch, G_AB_perfect,   G_BA_perfect)
loss_collapsed = cycle_consistency_loss(x_batch, G_AB_collapsed, G_BA_collapsed)
loss_noisy     = cycle_consistency_loss(x_batch, G_AB_noisy,     G_BA_noisy)

print(f"Perfect inverse generators : L_cyc = {loss_perfect:.6f}  (ideally 0)")
print(f"Collapsed generator        : L_cyc = {loss_collapsed:.4f}  (≈ E[|x|] = {np.mean(np.abs(x_batch)):.4f})")
print(f"Noisy approximate inverse  : L_cyc = {loss_noisy:.4f}")

# ── Visualise how loss varies with perturbation magnitude ────────────────────
sigmas     = np.linspace(0, 2, 50)
cyc_losses = []
for sigma in sigmas:
    G_fwd = lambda x, s=sigma: x + s
    G_bck = lambda x, s=sigma: x - s
    cyc_losses.append(cycle_consistency_loss(x_batch, G_fwd, G_bck))

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(sigmas, cyc_losses, color='#f59e0b', linewidth=2)
ax.set_xlabel('Shift magnitude (σ)')
ax.set_ylabel('Cycle consistency loss (L1)')
ax.set_title('Cycle loss vs generator perturbation magnitude', color='white')
plt.tight_layout()
plt.show()

**What to notice:** cycle consistency gives supervision **without paired data** — `G_BA(G_AB(x)) ≈ x`
pins the translators to be near-inverses, so content survives the round trip while style changes. The
loss is just an L1 between the input and its round trip; identity mappings trivially achieve 0, which
is why adversarial losses accompany it in the real CycleGAN.

## 4. Latent Diffusion — Compression Ratio

Stable Diffusion moves diffusion from pixel space into a compressed latent space.
The VAE reduces 512×512×3 pixels to 64×64×4 latents — a **49× reduction** in tensor size.
Each denoising step therefore operates on a much cheaper representation.

In [ ]:
# ── Compare tensor sizes: pixel-space vs latent-space diffusion ──────────────
configs_diffusion = [
    dict(name="Pixel-space (256×256)",  shape=(1, 3,   256, 256)),
    dict(name="Pixel-space (512×512)",  shape=(1, 3,   512, 512)),
    dict(name="Pixel-space (1024×1024)",shape=(1, 3,  1024,1024)),
    dict(name="Latent (SD 256→32)",     shape=(1, 4,    32,  32)),
    dict(name="Latent (SD 512→64)",     shape=(1, 4,    64,  64)),
    dict(name="Latent (SD 1024→128)",   shape=(1, 4,   128, 128)),
]

pixel_512 = 1 * 3 * 512 * 512

print(f"{'Configuration':<28} {'Elements':>10} {'vs pixel-512':>14} {'MB (float32)':>14}")
print("-" * 70)
for cfg in configs_diffusion:
    n = 1
    for s in cfg['shape']:
        n *= s
    ratio = pixel_512 / n
    mb    = n * 4 / 1e6  # float32 = 4 bytes
    print(f"{cfg['name']:<28} {n:>10,} {ratio:>13.1f}× {mb:>13.2f}")

print()
print(f"Compression ratio (512 pixel → 64 latent): {pixel_512 / (1*4*64*64):.0f}×")
print("Each denoising step over the latent is ~49× cheaper in memory.")

# ── Visualise relative sizes ──────────────────────────────────────────────────
labels  = [c['name'] for c in configs_diffusion]
sizes   = []
for cfg in configs_diffusion:
    n = 1
    for s in cfg['shape']:
        n *= s
    sizes.append(n)

colors = ['#f43f5e']*3 + ['#14b8a6']*3
fig, ax = plt.subplots(figsize=(10, 4))
bars = ax.barh(labels, sizes, color=colors, edgecolor='none')
ax.set_xlabel('Tensor elements (lower = cheaper per step)')
ax.set_title('Pixel-space vs Latent-space diffusion tensor sizes', color='white')
ax.set_xscale('log')
legend_handles = [
    mpatches.Patch(color='#f43f5e', label='Pixel-space'),
    mpatches.Patch(color='#14b8a6', label='Latent-space (SD)'),
]
ax.legend(handles=legend_handles)
plt.tight_layout()
plt.show()

**What to notice:** running diffusion in the VAE's latent (64×64×4) instead of pixels (512×512×3)
means each denoising step touches **~48× fewer values** — the single engineering decision that made
Stable Diffusion trainable and runnable on consumer GPUs. The VAE from lesson 3 is literally the
compressor in this pipeline.

## 5. Stable Diffusion — Tensor Shapes Through the Pipeline

The full Stable Diffusion inference pipeline:

1. **CLIP text encoder** — tokenized prompt → text embeddings (shape: `(77, 768)`)
2. **Start with latent noise** — `z_T ~ N(0, I)` (shape: `(1, 4, 64, 64)` for 512×512 output)
3. **Denoising loop** (20–50 DDIM steps): U-Net(z_t, t, text_emb) → predicted noise → update z
4. **VAE decoder** — decode final latent `z_0` → pixel image `(1, 3, 512, 512)`

In [ ]:
# ── Print all shapes through the SD pipeline ─────────────────────────────────
# No real model — just shape arithmetic

# Config
OUTPUT_RES   = 512     # target image resolution
VAE_FACTOR   = 8       # spatial downscale factor
LATENT_CH    = 4       # latent channels
CLIP_SEQ_LEN = 77      # CLIP max token length
CLIP_DIM     = 768     # CLIP text embedding dim  (ViT-L uses 1024)
N_STEPS      = 20      # DDIM steps
BATCH        = 1

latent_h = OUTPUT_RES // VAE_FACTOR   # 64
latent_w = OUTPUT_RES // VAE_FACTOR   # 64

stages = [
    ("Input prompt (tokens)",       (CLIP_SEQ_LEN,)),
    ("CLIP text embedding",         (CLIP_SEQ_LEN, CLIP_DIM)),
    ("Initial latent z_T ~ N(0,I)", (BATCH, LATENT_CH, latent_h, latent_w)),
]

print("── Stable Diffusion 512×512 pipeline (batch=1) ──")
print()
for name, shape in stages:
    n = 1
    for s in shape:
        n *= s
    print(f"  {name:<38} shape={str(shape):<28} elements={n:,}")

print()
print(f"  Denoising loop: {N_STEPS} DDIM steps")
for step in [1, N_STEPS // 2, N_STEPS]:
    z_shape = (BATCH, LATENT_CH, latent_h, latent_w)
    n = BATCH * LATENT_CH * latent_h * latent_w
    print(f"    Step {step:>2}: U-Net input  z_t shape={z_shape}  ({n:,} values)")
    print(f"    Step {step:>2}: U-Net output ε̂    shape={z_shape}  ({n:,} values)")

print()
final_stages = [
    ("Final latent z_0 (post-denoise)", (BATCH, LATENT_CH, latent_h, latent_w)),
    ("VAE decoder output (pixels)",     (BATCH, 3, OUTPUT_RES, OUTPUT_RES)),
]
for name, shape in final_stages:
    n = 1
    for s in shape:
        n *= s
    print(f"  {name:<38} shape={str(shape):<28} elements={n:,}")

print()
latent_n = BATCH * LATENT_CH * latent_h * latent_w
pixel_n  = BATCH * 3 * OUTPUT_RES * OUTPUT_RES
print(f"  Latent → pixel expansion ratio: {pixel_n / latent_n:.0f}× (VAE decoder)")
print(f"  Each denoising step works on {latent_n:,} values vs {pixel_n:,} in pixel-space")
print(f"  Compute savings per step: ~{pixel_n // latent_n}×")

**What to notice:** the full Stable Diffusion pipeline in shapes — text encoder → 77×768 prompt
tokens; U-Net denoising in the 4×64×64 latent, conditioned by cross-attention on those tokens (the
Q-from-image, K/V-from-text pattern of lesson 4); VAE decoder back to 3×512×512 pixels. Every
component is a lesson from this course composed into one system.

## Gotchas & tradeoffs

- **Patch scrambling is silent.** A wrong transpose order keeps all shapes valid — always test against
  explicit slicing (as above).
- **CFG has a sweet spot.** `w ≈ 7–8` for Stable Diffusion; too low ignores the prompt, too high
  fries colors and anatomy (extrapolation off-manifold). It also **doubles** inference cost (two
  forward passes per step).
- **Latent diffusion inherits its VAE's ceiling** — details the autoencoder can't reconstruct (fine
  text, faces at distance) are unrecoverable no matter how good the denoiser.
- **ViTs are data-hungry:** without convolution's built-in locality bias they underperform CNNs on
  small datasets and need large-scale pretraining or heavy augmentation.

In [ ]:
# CFG's hidden bill: two U-Net evaluations per step (conditional + unconditional)
steps = 50
print(f'{steps} sampling steps without CFG: {steps} U-Net forward passes')
print(f'{steps} sampling steps with    CFG: {2*steps} U-Net forward passes  (2x cost)')

# and the wrong transpose, demonstrated: shapes fine, patches scrambled
bad = img_t.reshape(n_side, P_t, n_side, P_t, 3).transpose(2, 0, 1, 3, 4).reshape(n_side*n_side, -1)
print('\nwrong transpose: shape', bad.shape, '== right shape', pt.shape,
      'but content matches?', np.allclose(bad, manual))

**What to notice:** the wrong transpose produces the *identical shape* but scrambled content — a bug
that passes every shape check and silently destroys spatial structure. And CFG's quality boost costs a
full 2× inference: every real deployment weighs guidance strength against latency and cost.

## Key takeaways

1. **ViT** converts an image into a sequence of patch tokens (e.g., 196 tokens for 224×224, P=16), prepends a [CLS] token, and runs a standard Transformer encoder — no convolutions needed.
2. **CFG formula** `(1+w)·ε_c − w·ε_∅` amplifies the guidance direction from unconditional toward conditional. w=7.5 is a common default; higher values trade diversity for prompt fidelity.
3. **CycleGAN** enforces `G_BA(G_AB(x)) ≈ x` via an L1 cycle loss, enabling unpaired image-to-image translation without matched training pairs.
4. **Stable Diffusion** compresses images 49× into latent space (512×512×3 → 64×64×4) with a pre-trained VAE, runs diffusion in that cheap latent space conditioned on CLIP text embeddings, then decodes the result — making high-resolution synthesis tractable.

---

## ✏️ Your turn

Work through the two exercises below.
Each has a stub to fill in, an assert cell that prints `✅` when your solution is correct,
and a solution hidden in a `<details>` block.

### Exercise 1 — Patch Embedding

Implement `extract_patches(image, patch_size)` which splits an image into non-overlapping patches.

Given an image of shape `(H, W, C)`:
- `num_patches = (H // patch_size) * (W // patch_size)`
- Each patch has shape `(patch_size, patch_size, C)`
- Return an array of shape `(num_patches, patch_size, patch_size, C)`

**Hint:** reshape to `(n_h, P, n_w, P, C)` then transpose to `(n_h, n_w, P, P, C)` and merge the first two axes.

In [ ]:
def extract_patches(image, patch_size):
    """
    Split image into non-overlapping square patches.

    Parameters
    ----------
    image      : (H, W, C) float array
    patch_size : int, P

    Returns
    -------
    patches : (num_patches, patch_size, patch_size, C)
    """
    # TODO(you): implement patch extraction
    # 1. Get H, W, C from image.shape
    # 2. Compute n_h = H // patch_size, n_w = W // patch_size
    # 3. Reshape and transpose to isolate each patch
    # 4. Return array of shape (n_h * n_w, patch_size, patch_size, C)
    ...

In [ ]:
# ── Assertions ────────────────────────────────────────────────────────────────
test_image = rng.normal(0, 1, (224, 224, 3)).astype(np.float32)
P = 16
patches = extract_patches(test_image, P)

expected_num = (224 // P) * (224 // P)
assert patches is not None, "extract_patches returned None — did you forget to return?"
assert patches.shape[0] == expected_num, \
    f"Expected {expected_num} patches, got {patches.shape[0]}"
assert patches.shape[1:] == (P, P, 3), \
    f"Expected each patch shape ({P}, {P}, 3), got {patches.shape[1:]}"

# Verify no data is lost: reconstructed image should match original
n_h = n_w = 224 // P
reconstructed = patches.reshape(n_h, n_w, P, P, 3)\
                        .transpose(0, 2, 1, 3, 4)\
                        .reshape(224, 224, 3)
assert np.allclose(reconstructed, test_image, atol=1e-6), \
    "Patches do not reconstruct the original image — check your reshape/transpose order"

# Smaller sanity-check image
tiny = np.arange(48, dtype=float).reshape(4, 4, 3)
tiny_patches = extract_patches(tiny, patch_size=2)
assert tiny_patches.shape == (4, 2, 2, 3), \
    f"For 4×4×3 image with P=2 expected shape (4, 2, 2, 3), got {tiny_patches.shape}"

print("✅ Exercise 1 passed")

<details>
<summary>💡 Show solution</summary>

```python
def extract_patches(image, patch_size):
    H, W, C = image.shape
    P = patch_size
    n_h, n_w = H // P, W // P

    # Interleave patch-grid dims with within-patch dims
    patches = image.reshape(n_h, P, n_w, P, C)   # (n_h, P, n_w, P, C)
    patches = patches.transpose(0, 2, 1, 3, 4)   # (n_h, n_w, P, P, C)
    return patches.reshape(n_h * n_w, P, P, C)   # (num_patches, P, P, C)
```

</details>

### Exercise 2 — Classifier-Free Guidance

Implement the CFG combination formula:

$$\hat{\varepsilon} = (1+w)\,\varepsilon_c - w\,\varepsilon_\emptyset$$

Special cases to verify:
- When $w = 0$: output equals `cond_output` ($(1+0)\cdot\varepsilon_c - 0\cdot\varepsilon_\emptyset = \varepsilon_c$)
- When $w$ is very large: the output is dominated by the guidance direction $w\cdot(\varepsilon_c - \varepsilon_\emptyset)$

**Note:** `(1+w)·ε_c - w·ε_∅` is algebraically identical to `ε_c + w·(ε_c - ε_∅)`. Both forms are correct.

In [ ]:
def cfg_guided(cond_output, uncond_output, guidance_scale):
    """
    Combine conditional and unconditional model outputs with CFG.

    Parameters
    ----------
    cond_output   : array-like, conditional model output ε_c
    uncond_output : array-like, unconditional model output ε_∅
    guidance_scale: float, w ≥ 0

    Returns
    -------
    combined output same shape as inputs
    """
    # TODO(you): implement the CFG formula
    # Hint: (1 + w) * cond - w * uncond
    ...

In [ ]:
# ── Assertions ────────────────────────────────────────────────────────────────
cond_vec   = np.array([1.0, 0.5, -0.3])
uncond_vec = np.array([0.2, 0.2,  0.2])

# w=0: output must equal cond_output
out_w0 = cfg_guided(cond_vec, uncond_vec, guidance_scale=0.0)
assert out_w0 is not None, "cfg_guided returned None — did you forget to return?"
assert np.allclose(out_w0, cond_vec, atol=1e-9), \
    f"At w=0, expected cond_output {cond_vec}, got {out_w0}"

# w=1: output = 2*cond - 1*uncond
out_w1 = cfg_guided(cond_vec, uncond_vec, guidance_scale=1.0)
expected_w1 = 2 * cond_vec - 1 * uncond_vec
assert np.allclose(out_w1, expected_w1, atol=1e-9), \
    f"At w=1, expected {expected_w1}, got {out_w1}"

# w=7.5: output = 8.5*cond - 7.5*uncond
out_w75 = cfg_guided(cond_vec, uncond_vec, guidance_scale=7.5)
expected_w75 = 8.5 * cond_vec - 7.5 * uncond_vec
assert np.allclose(out_w75, expected_w75, atol=1e-9), \
    f"At w=7.5, expected {expected_w75}, got {out_w75}"

# Large w: output must be dominated by (cond - uncond) direction
w_large    = 100.0
out_large  = cfg_guided(cond_vec, uncond_vec, guidance_scale=w_large)
guidance_dir = cond_vec - uncond_vec  # should dominate at large w
# Check that the large-w output points in the same direction as the guidance direction
cos_sim = np.dot(out_large, guidance_dir) / (np.linalg.norm(out_large) * np.linalg.norm(guidance_dir))
assert cos_sim > 0.99, \
    f"At large w, output should be nearly parallel to (cond-uncond). Cosine similarity: {cos_sim:.4f}"

# Shape preservation
batch_cond   = rng.normal(0, 1, (4, 8))
batch_uncond = rng.normal(0, 1, (4, 8))
out_batch = cfg_guided(batch_cond, batch_uncond, guidance_scale=7.5)
assert out_batch.shape == (4, 8), \
    f"Output shape should match input shape (4, 8), got {out_batch.shape}"

print("✅ Exercise 2 passed")

<details>
<summary>💡 Show solution</summary>

```python
def cfg_guided(cond_output, uncond_output, guidance_scale):
    w = guidance_scale
    return (1 + w) * np.asarray(cond_output) - w * np.asarray(uncond_output)
```

The formula can also be written as:

```python
    return np.asarray(cond_output) + w * (np.asarray(cond_output) - np.asarray(uncond_output))
```

Both are algebraically equivalent. The second form makes it clear that guidance adds
`w` times the *guidance direction* `(ε_c − ε_∅)` on top of the conditional output.

</details>